In [1]:
import glob
import pandas as pd

# Load files

In [2]:
metadata_files = glob.glob("outputs/extracted_metadata_*.jsonl")
metadata_files

['outputs/extracted_metadata_unhcr.jsonl']

In [3]:
log_files = glob.glob("outputs/extraction_calls_*.jsonl")
log_files

['outputs/extraction_calls_unhcr.jsonl']

# Metadata

In [4]:
df = pd.concat([pd.read_json(x, lines=True) for x in metadata_files]).reset_index(drop=True)
df

,image_name,image_path,source,metadata
0,1_advocacy_note_mineaction_-_niger_eng_figure_...,data/snapshots/1_advocacy_note_mineaction_-_ni...,unhcr,{'title': 'Graph2: Number of ED incidents per ...
1,1_advocacy_note_mineaction_-_niger_eng_figure_...,data/snapshots/1_advocacy_note_mineaction_-_ni...,unhcr,{'title': 'Graph2. Distribution of ED incident...
2,1_advocacy_note_mineaction_-_niger_eng_figure_...,data/snapshots/1_advocacy_note_mineaction_-_ni...,unhcr,"{'title': 'Graph3. Profil of ED victims', 'int..."
3,57319_figure_007.png,data/snapshots/57319_figure_007.png,unhcr,"{'title': 'Key findings', 'internal_identifier..."


In [5]:
df.loc[3, "metadata"]

{'title': 'Key findings',
 'internal_identifier': None,
 'subject_domain': 'Health',
 'subject_summary': 'Referral care coverage and referral volume for Syrian refugees in Lebanon.',
 'panel_title': None,
 'variable_name': 'Referral care coverage and referral volume',
 'category_dimension': None,
 'category_labels': None,
 'population_group': 'Syrian refugees',
 'time_period': 'January to December 2016',
 'temporal_granularity': 'monthly',
 'geographic_scope': 'Lebanon',
 'geographic_entities': ['Lebanon', 'Bekaa'],
 'geographic_granularity': 'country and subnational region',
 'geographic_role': ['primary geographic scope', 'subnational location'],
 'location_type': 'hospital',
 'unit_of_measure': 'percent and number of referrals',
 'currency': None,
 'measure_type': 'proportion and count',
 'comparison_group': ['2015', 'remaining 30 contracted hospitals'],
 'row_dimension': None,
 'column_dimension': None,
 'visualization_type': 'infographic',
 'data_source': ['UNHCR'],
 'source_docum

# Usage

In [6]:
# USD per 1M tokens
PRICING = {
    "gpt-5.5": {
        "input_per_1M": 5.00,
        "output_per_1M": 30.00,
    },
    "gpt-5.4": {
        "input_per_1M": 2.50,
        "output_per_1M": 15.00,
    },
    "gpt-5.4-mini": {
        "input_per_1M": 0.75,
        "output_per_1M": 4.50,
    },
}


def calculate_cost(row):
    model = row["model"]
    input_tokens = row["input_tokens"]
    output_tokens = row["output_tokens"]

    try:
        input_cost = PRICING.get(model).get("input_per_1M") * input_tokens / 1e6
        output_cost = PRICING.get(model).get("output_per_1M") * output_tokens / 1e6
        total_cost = input_cost + output_cost
    except:
        total_cost = None

    return total_cost

In [7]:
log_df = pd.concat([pd.read_json(x, lines=True) for x in log_files]).reset_index(
    drop=True
)
log_df["input_tokens"] = log_df["usage"].apply(lambda x: x.get("input_tokens"))
log_df["output_tokens"] = log_df["usage"].apply(lambda x: x.get("output_tokens"))
log_df["usd_cost"] = log_df.apply(lambda row: calculate_cost(row), axis=1)
log_df

,image_name,source,model,elapsed_seconds,usage,error,input_tokens,output_tokens,usd_cost
0,1_advocacy_note_mineaction_-_niger_eng_figure_...,unhcr,gpt-5.4-mini,9.211551,"{'input_tokens': 5789, 'input_tokens_details':...",NaN,5789,995,0.008819
1,1_advocacy_note_mineaction_-_niger_eng_figure_...,unhcr,gpt-5.4-mini,15.941243,"{'input_tokens': 6754, 'input_tokens_details':...",NaN,6754,1707,0.012747
2,1_advocacy_note_mineaction_-_niger_eng_figure_...,unhcr,gpt-5.4-mini,15.274902,"{'input_tokens': 5932, 'input_tokens_details':...",NaN,5932,1750,0.012324
3,57319_figure_007.png,unhcr,gpt-5.4-mini,17.739174,"{'input_tokens': 5766, 'input_tokens_details':...",NaN,5766,1907,0.012906


In [8]:
total_cost = log_df["usd_cost"].sum()
ave_cost = total_cost / len(log_df)

print(f"Total cost: ${total_cost:.2f}")
print(f"Ave cost per call: ${ave_cost:.4f}")

Total cost: $0.05
Ave cost per call: $0.0117
